# Dataset checker

Change `NAME` in the next cell, then **Run All**. Do it once for each dataset and compare the answers.

Each section says what you should see. Raw data comes from OpenML (IDs from `docs/dataset_register.md`).

In [3]:
NAME = "covertype"   # adult | diabetes | kick | bank | covertype

In [4]:
import json, hashlib
import numpy as np
import pandas as pd
import openml

# From docs/dataset_register.md: OpenML ID + how the raw target becomes y = 1
REGISTER = {
    "adult":    {"openml_id": 1590,  "positive": lambda t: t.astype(str).str.strip().str.rstrip(".") == ">50K"},
    "diabetes": {"openml_id": 4541,  "positive": lambda t: t.astype(str).str.strip() == "<30"},
    "kick":     {"openml_id": 41162, "positive": lambda t: t.astype(str).str.strip() == "1"},
    "bank":     {"openml_id": 1461,  "positive": lambda t: t.astype(str).str.strip() == "2"},
    "covertype":{"openml_id": 1596,  "positive": lambda t: t.astype(str).str.strip() == "2"},
}
reg = REGISTER[NAME]

## 1. Raw data (straight from OpenML)
Look at the original columns, their types, and how much is missing. Later sections check that the clean file matches it.

In [5]:
ds = openml.datasets.get_dataset(reg["openml_id"], download_data=True,
                                 download_qualities=False, download_features_meta_data=False)
raw_X, raw_t, _, _ = ds.get_data(dataset_format="dataframe", target=ds.default_target_attribute)
print("raw shape:", raw_X.shape, "| target column:", ds.default_target_attribute)
raw_X.head()

raw shape: (581012, 54) | target column: class


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type31,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40
0,2596.0,51.0,3,258.0,0.0,510.0,221,232,148,6279.0,...,0,0,0,0,0,0,0,0,0,0
1,2590.0,56.0,2,212.0,-6.0,390.0,220,235,151,6225.0,...,0,0,0,0,0,0,0,0,0,0
2,2804.0,139.0,9,268.0,65.0,3180.0,234,238,135,6121.0,...,0,0,0,0,0,0,0,0,0,0
3,2785.0,155.0,18,242.0,118.0,3090.0,238,238,122,6211.0,...,0,0,0,0,0,0,0,0,0,0
4,2595.0,45.0,2,153.0,-1.0,391.0,220,234,150,6172.0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
print("raw target values:")
print(raw_t.value_counts(dropna=False))

raw_y = reg["positive"](raw_t).astype(int)
print()
print("raw prevalence using the register rule:", round(raw_y.mean(), 4))

raw target values:
class
2    283301
1    211840
3     35754
7     20510
6     17367
5      9493
4      2747
Name: count, dtype: int64

raw prevalence using the register rule: 0.4876


In [7]:
# Missing % per column. '?' is counted as missing. Anything over 30% should be gone from the clean file.
miss = raw_X.replace("?", np.nan).isna().mean().sort_values(ascending=False)
miss[miss > 0].map("{:.2%}".format)

Series([], dtype: float64)

In [8]:
# Number of distinct values in each text column. Columns with more than 50 need special handling (check the prep script).
raw_X.select_dtypes(exclude="number").nunique().sort_values(ascending=False)

Wilderness_Area1    2
Wilderness_Area2    2
Wilderness_Area3    2
Wilderness_Area4    2
Soil_Type1          2
Soil_Type2          2
Soil_Type3          2
Soil_Type4          2
Soil_Type5          2
Soil_Type6          2
Soil_Type7          2
Soil_Type8          2
Soil_Type9          2
Soil_Type10         2
Soil_Type11         2
Soil_Type12         2
Soil_Type13         2
Soil_Type14         2
Soil_Type15         2
Soil_Type16         2
Soil_Type17         2
Soil_Type18         2
Soil_Type19         2
Soil_Type20         2
Soil_Type21         2
Soil_Type22         2
Soil_Type23         2
Soil_Type24         2
Soil_Type25         2
Soil_Type26         2
Soil_Type27         2
Soil_Type28         2
Soil_Type29         2
Soil_Type30         2
Soil_Type31         2
Soil_Type32         2
Soil_Type33         2
Soil_Type34         2
Soil_Type35         2
Soil_Type36         2
Soil_Type37         2
Soil_Type38         2
Soil_Type39         2
Soil_Type40         2
dtype: int64

## 2. Clean data

In [9]:
df = pd.read_csv(f"data/{NAME}_clean.csv")
s  = json.load(open(f"data/{NAME}_split.json"))
# Bank and Covertype use different key names; map them onto the Adult/Diabetes/Kick ones
for new, old in [("pool_idx", "train_indices"), ("test_idx", "test_indices"), ("csv_md5", "md5")]:
    s.setdefault(new, s.get(old))
s.setdefault("test_size", 0.30)
y  = df["y"]
X  = df.drop(columns="y")
print("clean shape:", df.shape)
df.head()

clean shape: (581012, 55)


,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f46,f47,f48,f49,f50,f51,f52,f53,f54,y
0,-1.297805,-0.935157,-1.482820,-0.053767,-0.796273,-1.180146,0.330743,0.439143,0.142960,3.246283,...,-0.315238,-0.290284,-0.05273,-0.057143,-0.014313,-0.022653,-0.165956,-0.156014,-0.123654,0
1,-1.319235,-0.890480,-1.616363,-0.270188,-0.899197,-1.257106,0.293388,0.590899,0.221342,3.205504,...,-0.315238,-0.290284,-0.05273,-0.057143,-0.014313,-0.022653,-0.165956,-0.156014,-0.123654,0
2,-0.554907,-0.148836,-0.681563,-0.006719,0.318742,0.532212,0.816364,0.742654,-0.196691,3.126965,...,-0.315238,-0.290284,-0.05273,-0.057143,-0.014313,-0.022653,-0.165956,-0.156014,-0.123654,1
3,-0.622768,-0.005869,0.520322,-0.129044,1.227908,0.474492,0.965786,0.742654,-0.536343,3.194931,...,-0.315238,-0.290284,-0.05273,-0.057143,-0.014313,-0.022653,-0.165956,-0.156014,-0.123654,1
4,-1.301377,-0.988770,-1.616363,-0.547771,-0.813427,-1.256464,0.293388,0.540313,0.195215,3.165479,...,-0.315238,-0.290284,-0.05273,-0.057143,-0.014313,-0.022653,-0.165956,-0.156014,-0.123654,0


## 3. Raw vs clean
- Same number of rows (the pipeline never drops rows)
- Same prevalence
- y matches the raw target **row by row**. Catches shuffled rows.

In [10]:
print("rows   raw:", len(raw_X), " clean:", len(df), "->", len(raw_X) == len(df))
print("prev   raw:", round(raw_y.mean(), 4), " clean:", round(y.mean(), 4))
print("y identical row by row:", (raw_y.values == y.values).all())

rows   raw: 581012  clean: 581012 -> True
prev   raw: 0.4876  clean: 0.4876
y identical row by row: True


## 4. Format
Should see: 0 NaN, only float64 (y may be int), y only 0/1, columns f1..fk.

In [11]:
print("NaN count:", X.isna().sum().sum())
print("dtypes:", X.dtypes.value_counts().to_dict())
print("y values:", sorted(y.unique()))
print("columns are f1..fk:", list(X.columns) == [f"f{i}" for i in range(1, X.shape[1] + 1)])

NaN count: 0
dtypes: {dtype('float64'): 54}
y values: [np.int64(0), np.int64(1)]
columns are f1..fk: True


## 5. Standardisation
Every mean should be ~0 and every std ~1. The table shows the worst columns.

In [12]:
stats = pd.DataFrame({"mean": X.mean(), "std": X.std(ddof=0)})
print("max |mean|   :", stats["mean"].abs().max())
print("max |std - 1|:", (stats["std"] - 1).abs().max())
stats.sort_values("std").head()

max |mean|   : 4.977854418906859e-16
max |std - 1|: 3.3306690738754696e-16


,mean,std
f11,5.009162e-17,1.0
f17,2.974190e-17,1.0
f16,3.756871e-17,1.0
f6,0.000000e+00,1.0
f28,3.130726e-17,1.0


## 6. Frozen split
Should see: 0 overlap, every row used once, test is 30%, all three prevalences the same, MD5 True.

**MD5 False** means this CSV is not the file the split was made from. Rebuild it with the prep script.

In [13]:
pool, test = np.array(s["pool_idx"]), np.array(s["test_idx"])
print("seed:", s["seed"])
print("overlap:", len(set(pool) & set(test)))
print("every row used once:", sorted(np.r_[pool, test].tolist()) == list(range(len(df))))
print("test fraction:", round(len(test) / len(df), 4))
print("prevalence  all / pool / test:", round(y.mean(), 4), round(y.iloc[pool].mean(), 4), round(y.iloc[test].mean(), 4))
md5 = hashlib.md5(open(f"data/{NAME}_clean.csv", "rb").read()).hexdigest()
print("CSV md5 matches split json:", md5 == s["csv_md5"])

seed: 332
overlap: 0
every row used once: True
test fraction: 0.3
prevalence  all / pool / test: 0.4876 0.4876 0.4876
CSV md5 matches split json: False


### 6b. Can the split be rebuilt?
Run the same `train_test_split` again with the saved seed. If the stored indices are identical, the split really came from this data and seed, and nobody edited it by hand.

In [14]:
from sklearn.model_selection import train_test_split
re_pool, re_test = train_test_split(np.arange(len(y)), test_size=s["test_size"],
                                    random_state=s["seed"], stratify=y)
print("pool identical:", sorted(re_pool.tolist()) == sorted(s["pool_idx"]))
print("test identical:", sorted(re_test.tolist()) == sorted(s["test_idx"]))

pool identical: True
test identical: True


### 6c. Do pool and test look alike?
A random split should give the two halves the same feature distribution. For each feature, the gap between the pool mean and the test mean should be tiny (well under 0.05, since features have SD 1). A large gap means the split is not random.

In [15]:
gap = (X.iloc[pool].mean() - X.iloc[test].mean()).abs().sort_values(ascending=False)
print("largest pool-vs-test mean gap:", round(gap.iloc[0], 4))
gap.head(5)

largest pool-vs-test mean gap: 0.0067


f38    0.006694
f20    0.006005
f25    0.005442
f41    0.005275
f14    0.005123
dtype: float64

## 7. Leakage and duplicates
Correlation above ~0.9 likely means the answer leaked into a feature. 0.1â€“0.4 is normal.

In [16]:
print("duplicate feature columns:", pd.util.hash_pandas_object(X.T, index=False).duplicated().sum())
print("duplicate rows:", X.duplicated().sum())
X.corrwith(y).abs().sort_values(ascending=False).head(10)

duplicate feature columns: 0
duplicate rows: 0


f14    0.211598
f26    0.197150
f52    0.146113
f53    0.144098
f10    0.138394
f1     0.133892
f11    0.131765
f43    0.131293
f36    0.130681
f54    0.111237
dtype: float64

## 8. Summary row
Copy this into a table for each dataset and compare.

In [17]:
pd.DataFrame([{
    "dataset": NAME, "rows": len(df), "features": X.shape[1], "prevalence": round(y.mean(), 4),
    "raw_rows_match": len(raw_X) == len(df), "y_matches_raw": (raw_y.values == y.values).all(),
    "md5_ok": md5 == s["csv_md5"], "seed": s["seed"], "max_corr": round(X.corrwith(y).abs().max(), 3),
}])

,dataset,rows,features,prevalence,raw_rows_match,y_matches_raw,md5_ok,seed,max_corr
0,covertype,581012,54,0.4876,True,True,False,332,0.212
